In [31]:
import os
import pandas as pd
import json
import re

# Funktion zum Extrahieren von Zahlen für die richtige Sortierung
def sort_instances(instance_name):
    numbers = re.findall(r'\d+', instance_name)
    return tuple(int(num) for num in numbers)  # Zahlen extrahieren und als Tupel zurückgeben

# Basispfad (aktueller Ordner des Jupyter Notebooks)
solution_path = os.getcwd()

# Benutzerdefinierte Reihenfolge für Strategien
strategy_order = {"costs": 0, "weighted": 1, "hierarchical": 2, "hierarchical_tolerance": 3}

# Dictionaries für die getrennten Daten (3 und 6 Objectives)
data_3_objectives = []
data_6_objectives = []

# Ordner rekursiv durchlaufen
for root, dirs, files in os.walk(solution_path):
    for file in files:
        if file.endswith(".json"):  # Nur JSON-Dateien berücksichtigen
            
            if file.startswith("Solution"):  # Nur Ergebnisdateien berücksichtigen
                continue
        
            file_path = os.path.join(root, file)
            
            # JSON-Datei einlesen mit Fehlerbehandlung
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
            except json.JSONDecodeError:
                print(f"Ungültige JSON-Datei übersprungen: {file_path}")
                continue
            
            # Daten extrahieren
            instance = data.get("instance", "N/A")
            computational_time = data.get("computational_time", None)
            strategy = os.path.basename(os.path.dirname(file_path))
            objectives_type = os.path.basename(os.path.dirname(os.path.dirname(file_path)))
            
            # Ergebnisse aus der "results"-Liste extrahieren
            result_dict = {"Instance": instance, "Strategy": strategy, "Computational_Time": computational_time}
            for result in data.get("results", []):
                objective_name = result.get("Objective", "N/A")
                objective_value = result.get("Value", None)
                result_dict[objective_name] = objective_value
            
            # Daten je nach Objective Type speichern
            if objectives_type == "3_Objectives":
                data_3_objectives.append(result_dict)
            elif objectives_type == "6_Objectives":
                data_6_objectives.append(result_dict)

# Daten in DataFrames umwandeln
df_3_objectives = pd.DataFrame(data_3_objectives)
df_6_objectives = pd.DataFrame(data_6_objectives)

# Hilfsspalte für die Sortierung erstellen
df_3_objectives["Instance_Sort"] = df_3_objectives["Instance"].map(sort_instances)
df_6_objectives["Instance_Sort"] = df_6_objectives["Instance"].map(sort_instances)

# Strategiereihenfolge hinzufügen
df_3_objectives["Strategy_Order"] = df_3_objectives["Strategy"].map(strategy_order)
df_6_objectives["Strategy_Order"] = df_6_objectives["Strategy"].map(strategy_order)

# Sortieren: Erst nach Instanz, dann nach der benutzerdefinierten Strategiereihenfolge
df_3_objectives = df_3_objectives.sort_values(by=["Instance_Sort", "Strategy_Order"]).drop(columns=["Instance_Sort", "Strategy_Order"]).reset_index(drop=True)
df_6_objectives = df_6_objectives.sort_values(by=["Instance_Sort", "Strategy_Order"]).drop(columns=["Instance_Sort", "Strategy_Order"]).reset_index(drop=True)

# Formatierung: Instance nur einmal anzeigen
def format_table(df):
    df["Instance"] = df["Instance"].mask(df["Instance"].duplicated(), "")  # Leer lassen, wenn die Instanz schon angezeigt wurde
    return df

# Formatieren
df_3_objectives = format_table(df_3_objectives)
df_6_objectives = format_table(df_6_objectives)

In [32]:
df_3_objectives

,Instance,Strategy,Computational_Time,Construction Fulfillment,Non-Regular Driver Usage,Worker Work Distance
0,a3_o80_m10_an10_ar9_reduced,costs,7.698,3,42,3620.910238
1,,weighted,19.480,3,38,3844.726950
2,,hierarchical,44.734,3,38,3844.726950
3,,hierarchical_tolerance,38.191,3,46,3473.542393
4,a5_o96_m10_an10_ar10_reduced,costs,16.265,5,14,5147.983804
5,,weighted,46.042,5,1,6109.324232
6,,hierarchical,131.787,5,1,6109.324232
7,,hierarchical_tolerance,96.556,5,11,5198.383302
8,a10_o107_m5_an57_ar12,costs,57.365,9,16,4304.312645
9,,weighted,201.223,9,2,5018.159642


In [33]:
df_6_objectives

,Instance,Strategy,Computational_Time,Construction Fulfillment,Non-Regular Driver Usage,Worker Work Distance,Machine Transport Distance,Machine Usage,Worker Usage
0,a3_o80_m10_an10_ar9_reduced,costs,3080.952,3,40,4635.594580,70.585534,2,6
1,,weighted,808.781,2,8,2697.742155,29.732137,2,5
2,,hierarchical,1296.713,3,38,3844.726950,917.385600,2,8
3,,hierarchical_tolerance,273.293,3,46,3811.448838,99.194357,2,8
4,a5_o96_m10_an10_ar10_reduced,costs,97.908,5,10,6432.282747,95.862978,4,8
5,,weighted,238.739,5,1,6242.385287,122.263735,5,9
6,,hierarchical,1242.009,5,1,6109.324232,191.725955,6,9
7,,hierarchical_tolerance,10794.695,5,11,5706.335094,26.400758,6,9
8,a10_o107_m5_an57_ar12,costs,73.580,9,22,3844.799956,358.168841,4,7
9,,weighted,3109.467,9,2,5632.574529,296.222068,5,8
